#  Ingestão do data lake com boto3 

In [0]:

# Os três passos de toda ingestão:
# 1. **extrair** · o `boto3` lista e baixa cada arquivo do bucket `DataLakeVertice`
#  2. **pousar** · cada arquivo vai para o volume, no mesmo caminho: `/Volumes/lakehouse_vertice/bronze/raw/`
#  3. **carregar** · o mesmo `read_files` de ontem transforma o raw nas 9 tabelas da bronze
# MAGIC
# MAGIC Rode com a computação **serverless**, uma célula de cada vez.
# MAGIC
# MAGIC ## Configuração: as chaves chegam pelo ambiente
# MAGIC
# MAGIC A chave **não fica escrita no código**. O notebook lê tudo de `os.environ`, as variáveis de ambiente,
# MAGIC como em qualquer servidor ou na própria AWS. Quem preenche os valores é o **job**: cada parâmetro do job
# MAGIC chega ao notebook como um *widget* com o mesmo nome, e a célula abaixo passa os widgets para o ambiente.
# MAGIC
# MAGIC - **Rodando na mão:** aparecem 4 caixinhas no topo do notebook. Cole os valores da tabela de conexão.
# MAGIC - **Rodando pelo job:** os valores vêm dos **Parameters** da tarefa (etapa 4). Ninguém digita nada.
# MAGIC - **Ou pelo campo Environment variables (Beta)** da tarefa: aí os valores já chegam no `os.environ`, e a
# MAGIC   célula usa direto, sem caixinha.
# MAGIC
# MAGIC Os nomes `AWS_ACCESS_KEY_ID` e `AWS_SECRET_ACCESS_KEY` são o padrão da AWS: o `boto3` encontra as chaves
# MAGIC sozinho no ambiente, sem você passar no código.

# COMMAND ----------

import os

VARIAVEIS = ["S3_ENDPOINT", "AWS_DEFAULT_REGION", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"]

for nome in VARIAVEIS:
    if os.environ.get(nome):                       # já veio do job (Environment variables)? usa esse
        continue
    dbutils.widgets.text(nome, "")                 # senão, a caixinha (o job preenche pelos Parameters)
    os.environ[nome] = dbutils.widgets.get(nome)   # e o valor vira variável de ambiente

faltando = [nome for nome in VARIAVEIS if not os.environ[nome]]
assert not faltando, f"preencha: {faltando}"      # para aqui se faltar alguma chave

S3_BUCKET = "DataLakeVertice"
RAW = "/Volumes/lakehouse_vertice/bronze/raw"
print("ambiente pronto:", os.environ["S3_ENDPOINT"])


ambiente pronto: https://elgbwtelwlxjidjbxsrm.storage.supabase.co/storage/v1/s3


## Diagnóstico da conexão

In [0]:
import os, boto3
from botocore.config import Config

# diagnostico: o que a sessao realmente tem (sem exibir as chaves)
for nome in ["S3_ENDPOINT", "AWS_DEFAULT_REGION", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"]:
    v = os.environ.get(nome, "")
    print(f"{nome}: len={len(v)} http={v.startswith('http')} region={v == 'us-east-1'}")

try:
    s3 = boto3.client(
        "s3",
        endpoint_url=os.environ["S3_ENDPOINT"],
        config=Config(s3={"addressing_style": "path"}),
    )
    print("buckets:", [b["Name"] for b in s3.list_buckets()["Buckets"]])
except Exception as e:
    r = getattr(e, "response", {})
    print("tipo:", type(e).__name__)
    print("HTTP:", r.get("ResponseMetadata", {}).get("HTTPStatusCode"))
    print("Error:", r.get("Error"))
    print("headers:", r.get("ResponseMetadata", {}).get("HTTPHeaders"))

S3_ENDPOINT: len=62 http=True region=False
AWS_DEFAULT_REGION: len=9 http=False region=True
AWS_ACCESS_KEY_ID: len=32 http=False region=False
AWS_SECRET_ACCESS_KEY: len=64 http=False region=False
buckets: ['DataLakeVertice']


## Passo 1 · Conectar no data lake

In [0]:

# Repare: nenhuma chave no código. O `boto3` lê `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` e
# `AWS_DEFAULT_REGION` do ambiente; só o endereço do Supabase vai explícito.

# COMMAND ----------

import boto3
from botocore.config import Config

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["S3_ENDPOINT"],
    config=Config(s3={"addressing_style": "path"}),   # o Supabase só entende o bucket no caminho da URL
)

for bucket in s3.list_buckets()["Buckets"]:
    print(bucket["Name"])



DataLakeVertice
clickup       25 arquivos
compras        1 arquivos
documentos     9 arquivos
excel         20 arquivos
financeiro     1 arquivos
msproject     15 arquivos
pmo            2 arquivos
rh             1 arquivos
timesheet      1 arquivos
total         75
apagada: /Volumes/lakehouse_vertice/bronze/raw/clickup
apagada: /Volumes/lakehouse_vertice/bronze/raw/compras
apagada: /Volumes/lakehouse_vertice/bronze/raw/documentos
apagada: /Volumes/lakehouse_vertice/bronze/raw/excel
apagada: /Volumes/lakehouse_vertice/bronze/raw/financeiro
apagada: /Volumes/lakehouse_vertice/bronze/raw/msproject
apagada: /Volumes/lakehouse_vertice/bronze/raw/pmo
apagada: /Volumes/lakehouse_vertice/bronze/raw/rh
apagada: /Volumes/lakehouse_vertice/bronze/raw/timesheet


## Passo 2 · O que tem no bucket?

In [0]:

## Cada arquivo é um objeto com uma **Key** (o caminho) e um **Size**. O S3 devolve no máximo 1.000 por página; o *paginator* busca todas as páginas.

# COMMAND ----------

objetos = []
for pagina in s3.get_paginator("list_objects_v2").paginate(Bucket=S3_BUCKET):
    objetos += pagina.get("Contents", [])

# só arquivos (o storage cria um objeto vazio de marcação em cada pasta)
arquivos = [o["Key"] for o in objetos if not o["Key"].endswith("/") and o["Size"] > 0
            and not o["Key"].endswith(".emptyFolderPlaceholder")]

from collections import Counter
for pasta, n in sorted(Counter(k.split("/")[0] for k in arquivos).items()):
    print(f"{pasta:<12} {n:>3} arquivos")
print(f"{'total':<12} {len(arquivos):>3}")


clickup       25 arquivos
compras        1 arquivos
documentos     9 arquivos
excel         20 arquivos
financeiro     1 arquivos
msproject     15 arquivos
pmo            2 arquivos
rh             1 arquivos
timesheet      1 arquivos
total         75


## Passo 3 · Apague o trabalho manual de ontem

In [0]:

# MAGIC É proposital: as pastas do raw e as 9 tabelas da bronze somem, e voltam vindas do data lake.

TABELAS = ["projetos", "status_semanal", "excel_tarefas", "clickup_tarefas", "msproject_tarefas",
           "colaboradores", "apontamentos", "custos", "entregas_fornecedor"]

for tabela in TABELAS:
    spark.sql(f"DROP TABLE IF EXISTS lakehouse_vertice.bronze.{tabela}")

for pasta in sorted({k.split("/")[0] for k in arquivos}):
    dbutils.fs.rm(f"{RAW}/{pasta}", True)
    print("apagada:", f"{RAW}/{pasta}")


## Passo 4 · Extrair e pousar: cada arquivo do bucket vai para o raw

In [0]:

import os

def baixar(key):
    destino = f"{RAW}/{key}"           # o mesmo caminho do bucket
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    conteudo = s3.get_object(Bucket=S3_BUCKET, Key=key)["Body"].read()   # 1 · extrair
    with open(destino, "wb") as f:                                        # 2 · pousar
        f.write(conteudo)
    return len(conteudo)

total = 0
for key in arquivos:
    total += baixar(key)
print(f"{len(arquivos)} arquivos, {total/1e6:.1f} MB no raw")

75 arquivos, 6.9 MB no raw


## Passo 5 · Carregar: o raw vira a bronze com  SQL 

In [0]:
%sql

-- Tudo como texto, sem limpar nada: a bronze guarda o que chegou. A limpeza é amanhã, na silver.



CREATE OR REPLACE TABLE lakehouse_vertice.bronze.projetos AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/pmo/projetos.csv',
                format => 'csv', header => true, inferColumnTypes => false);

CREATE OR REPLACE TABLE lakehouse_vertice.bronze.status_semanal AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/pmo/status_semanal.csv',
                format => 'csv', header => true, inferColumnTypes => false);

CREATE OR REPLACE TABLE lakehouse_vertice.bronze.colaboradores AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/rh/colaboradores.csv',
                format => 'csv', header => true, inferColumnTypes => false);

CREATE OR REPLACE TABLE lakehouse_vertice.bronze.apontamentos AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/timesheet/apontamentos.csv',
                format => 'csv', header => true, inferColumnTypes => false);

CREATE OR REPLACE TABLE lakehouse_vertice.bronze.custos AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/financeiro/custos.csv',
                format => 'csv', header => true, inferColumnTypes => false);

CREATE OR REPLACE TABLE lakehouse_vertice.bronze.entregas_fornecedor AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/compras/entregas_fornecedor.csv',
                format => 'csv', header => true, inferColumnTypes => false);


CREATE OR REPLACE TABLE lakehouse_vertice.bronze.excel_tarefas AS
SELECT `ID`               AS id,
       `Fase`             AS fase,
       `Tarefa`           AS tarefa,
       `Responsável`      AS responsavel,
       `Início previsto`  AS inicio_previsto,
       `Término previsto` AS termino_previsto,
       `Início real`      AS inicio_real,
       `Término real`     AS termino_real,
       `% Concluído`      AS pct_concluido,
       `Horas previstas`  AS horas_previstas,
       url_decode(_metadata.file_name) AS arquivo
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/excel/',
                format => 'excel', headerRows => 1, inferColumnTypes => false);

CREATE OR REPLACE TABLE lakehouse_vertice.bronze.clickup_tarefas AS
SELECT list.id   AS list_id,
       list.name AS list_name,
       t.*,
       url_decode(_metadata.file_name) AS arquivo
FROM (
    SELECT list, explode(tasks) AS t, _metadata
    FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/clickup/',
                    format => 'json', multiLine => true)
);

CREATE OR REPLACE TABLE lakehouse_vertice.bronze.msproject_tarefas AS
SELECT * EXCEPT (_rescued_data),
       url_decode(_metadata.file_name) AS arquivo
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/msproject/',
                format => 'xml', rowTag => 'Task');



num_affected_rows,num_inserted_rows


## Passo 6 · Validação tamanho dos arquivos

In [0]:
%sql
-- Deve aparecer :
-- apontamentos 30.883 · custos 23.640 · status_semanal 3.911 · clickup_tarefas 2.119 · excel_tarefas 1.663
-- msproject_tarefas 1.260 · entregas_fornecedor 640 · colaboradores 523 · projetos 60
SELECT 'projetos' AS tabela, COUNT(*) AS linhas FROM lakehouse_vertice.bronze.projetos
UNION ALL SELECT 'status_semanal', COUNT(*) FROM lakehouse_vertice.bronze.status_semanal
UNION ALL SELECT 'excel_tarefas', COUNT(*) FROM lakehouse_vertice.bronze.excel_tarefas
UNION ALL SELECT 'clickup_tarefas', COUNT(*) FROM lakehouse_vertice.bronze.clickup_tarefas
UNION ALL SELECT 'msproject_tarefas', COUNT(*) FROM lakehouse_vertice.bronze.msproject_tarefas
UNION ALL SELECT 'colaboradores', COUNT(*) FROM lakehouse_vertice.bronze.colaboradores
UNION ALL SELECT 'apontamentos', COUNT(*) FROM lakehouse_vertice.bronze.apontamentos
UNION ALL SELECT 'custos', COUNT(*) FROM lakehouse_vertice.bronze.custos
UNION ALL SELECT 'entregas_fornecedor', COUNT(*) FROM lakehouse_vertice.bronze.entregas_fornecedor
ORDER BY linhas DESC

tabela,linhas
apontamentos,30883
custos,23640
status_semanal,3911
clickup_tarefas,2119
excel_tarefas,1663
msproject_tarefas,1260
entregas_fornecedor,640
colaboradores,523
projetos,60
